# Gilbert-Varshamov bound / instance-sizing sanity checks (Strategy 03)

Checks the Gilbert-Varshamov bound formula and Hamming-ball-volume helper against the actual behavior of the prediction-and-repair framework (`Code/core/`), to sanity-check what target weights are actually achievable/reasonable for a given (n, k, q) before running a bigger sweep.


In [ ]:
import os
import sys

# instances_generator.py / LEP_prediction_and_repair_v2.py now live in
# Code/core/ (this notebook was relocated during the Aug 2026 folder
# reorganization), so it needs to be added to sys.path explicitly. Tries
# a few relative depths so this works whether Jupyter's cwd is this
# notebook's own folder or the Code/ root.
for _rel in ['../../core', '../core', 'core']:
    if os.path.isdir(_rel):
        sys.path.insert(0, os.path.abspath(_rel))


In [14]:
from sage.all import binomial, GF

def hamming_ball_volume(n, r, q):
    """
    Computes the size/volume of a Hamming ball of radius r in F_q^n.
    
    V_q(n, r) = sum_{j=0}^{r} binomial(n, j) * (q - 1)^j
    """
    return sum(binomial(n, j) * ((q - 1)**j) for j in range(r + 1))


def gilbert_varshamov_bound(n, k, q):
    """
    Computes the Gilbert-Varshamov distance d_GV for an [n, k] code over F_q.
    
    Returns the smallest integer d such that:
    V_q(n, d-1) >= q^(n - k)
    """
    target = q**(n - k)
    for d in range(1, n + 1):
        if hamming_ball_volume(n, d - 1, q) >= target:
            return d
    return n

In [22]:
from instances_generator import (
    generate_noisy_LCE_instance_CBA_bit_flip_version,
    obtain_parity_check_matrix,
)

from LEP_prediction_and_repair_v2 import (
    compute_posterior_table,
    monomial_approximation,
    build_active_row_lists,
    structured_sd_repair,
    prediction_and_repair_framework,
)

# exampel parameters
n, k, q = 64, 32, 17
alpha, beta = 0.01, 0.1
F = GF(q)

# generate noisy LEP instance
G1, G2, Q, Q_noisy = generate_noisy_LCE_instance_CBA_bit_flip_version(
    n, k, q, alpha, beta, is_monomial=True,
)

# turn the noisy hint into a BBLM posterior table
posterior_table = compute_posterior_table(Q_noisy, alpha, beta, is_permutation=False)

# build the monomial approximation Q_hat
Q_hat, S, D_loc, pi = monomial_approximation(posterior_table, F)

# print("Secret monomial matrix Q:")
# print(Q)
# print("\nNoisy hint for Q:")
# print(Q_noisy)
# print("\nMonomial approximation Q_hat:")
# print(Q_hat)
print("\nNumber of correctly approximated rows:",
        sum(1 for i in range(n) if list(Q_hat[i]) == list(Q[i])), "/", n)

# run the prediction-and-repair framework
H2 = obtain_parity_check_matrix(G2)
rmax = 2 # mirar hasta la mitad de n

def repair_fn(w_tilde, v):
    A, L = build_active_row_lists(v, S, D_loc, F, budgets=3)
    return structured_sd_repair(w_tilde, v, H2, Q_hat, A, L, rmax)

# w de gv *1.2
d_gv = gilbert_varshamov_bound(n, k, q)
target_w = max(1, int(round(d_gv * 1.2)))

print(f"Calculated GV Distance d_GV: {d_gv}")
print(f"Target Weight (1.2 * d_GV): {target_w}")

# --- Pass target_w to prediction_and_repair_framework ---
pairs = prediction_and_repair_framework(
    G1, G2, Q_hat, m_pair=2, n_iter=500, repair_fn=repair_fn,
    enum_params={'target_weight': target_w, 'max_trials': 500},
)

print("\nRecovered candidate equivalent codeword pairs:")
for v, w in pairs:
    print("v =", v, " w =", w)

# check if the recovered pairs are indeed equivalent codewords
# i.e. verify that w == v * Q under the TRUE secret Q (not just that
# w happens to satisfy H2 * w^T = 0, which structured_sd_repair already
# guarantees by construction).
print("\nVerifying recovered pairs against the true secret Q:")
all_correct = True
for idx, (v, w) in enumerate(pairs):
    w_true = v * Q
    is_correct = (w == w_true)
    all_correct = all_correct and is_correct

    print(f"Pair {idx}: v = {v}")
    print(f"          w (recovered) = {w}")
    print(f"          w (true, v*Q) = {w_true}")
    print(f"          equivalent under Q: {is_correct}")

print("\nAll recovered pairs equivalent under the true secret Q:", all_correct)

### THINGS I HAVE TO CHECK
# 1. Create tests


Number of correctly approximated rows: 24 / 64
Calculated GV Distance d_GV: 21
Target Weight (1.2 * d_GV): 25


KeyboardInterrupt: 